In [4]:
import numpy as np
import pyomo 
from pyomo.contrib.edi import Formulation
from pyomo.environ import units
f = Formulation()


x = f.Variable(name='x', guess=1.0, units='m', description='x-coordinate')
y = f.Variable(name='y', guess=1.0, units='m', description='y-coordinate')
z = f.Variable(name='z', guess=1.0, units='m', description='time')
u = f.Variable(name='u', guess=1.0, units='m', description='velocity')
v = f.Variable(name='v', guess=1.0, units='m', description='velocity')


f.Objective(z)
# f.Constraint(1 <= x-y)
# # f.Constraint(x >=1)
# f.Constraint(y >= 0.5)

# Constraints = [1*units.ft <= x-y + 2*units.second,
#                y >= 0.5*units.ft ]

Constraints = [z >= x + y + u + v + 1*units.second, ]

f.ConstraintList(Constraints)

# f.pprint()

from edi.solvers.solver import cvxopt_solve
res = cvxopt_solve(f)
print(res)
print(res['x'])



ValueError: Function cannot handle mismatching units in SumNode

In [4]:
# ===========
# Description
# ===========
# A simple aircraft sizing problem, formulated as a Geometric Program
# From:  Hoburg and Abbeel
#        Geometric Programming for Aircraft Design Optimization
#        AIAA Journal
#        2014

# =================
# Import Statements
# =================
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import units
from edi.objects.formulation import Formulation
from edi.objects.blackBoxFunctionModel import BlackBoxFunctionModel

# ===================
# Declare Formulation
# ===================
f = Formulation()

# =================
# Declare Variables
# =================

A   = f.Variable(name="A"   , guess = 10.0     , units = "-"   , description="aspect ratio")
C_D = f.Variable(name="C_D" , guess = 0.025    , units = "-"   , description="Drag coefficient of wing")
C_f = f.Variable(name="C_f" , guess = 0.003    , units = "-"   , description="skin friction coefficient")
C_L = f.Variable(name="C_L" , guess = 0.5      , units = "-"   , description="Lift coefficient of wing")
D   = f.Variable(name="D"   , guess = 300      , units = "N"   , description="total drag force")
Re  = f.Variable(name="Re"  , guess = 3e6      , units = "-"   , description="Reynold's number")
S   = f.Variable(name="S"   , guess = 10.0     , units = "ft^2" , description="total wing area")
V   = f.Variable(name="V"   , guess = 30.0     , units = "m/s" , description="cruising speed")
W   = f.Variable(name="W"   , guess = 10000.0  , units = "N"   , description="total aircraft weight")
W_w = f.Variable(name="W_w" , guess = 2500     , units = "N"   , description="wing weight")

# =================
# Declare Constants
# =================
C_Lmax      = f.Constant( name="C_Lmax"  , value=2.0     , units="-"      , description="max CL with flaps down")
CDA0        = f.Constant( name="CDA0"    , value=0.0306  , units="m^2"    , description="fuselage drag area")
e           = f.Constant( name="e"       , value=0.96    , units="-"      , description="Oswald efficiency factor")
k           = f.Constant( name="k"       , value=1.2     , units="-"      , description="form factor")
mu          = f.Constant( name="mu"      , value=1.78e-5 , units="kg/m/s" , description="viscosity of air")
N_ult       = f.Constant( name="N_ult"   , value=2.5     , units="-"      , description="ultimate load factor")
rho         = f.Constant( name="rho"     , value=1.23    , units="kg/m^3" , description="density of air")
S_wetratio  = f.Constant( name="Srat"    , value=2.05    , units="-"      , description="wetted area ratio")
tau         = f.Constant( name="tau"     , value=0.12    , units="-"      , description="airfoil thickness to chord ratio")
V_min       = f.Constant( name="V_min"   , value=22      , units="m/s"    , description="takeoff speed")
W_0         = f.Constant( name="W_0"     , value=4940.0  , units="N"      , description="aircraft weight excluding wing")
W_W_coeff1  = f.Constant( name="W_c1"    , value=8.71e-5 , units="1/m"    , description="Wing Weight Coefficient 1")
W_W_coeff2  = f.Constant( name="W_c2"    , value=45.24   , units="Pa"     , description="Wing Weight Coefficient 2" )

# =====================
# Declare the Objective
# =====================
f.Objective(D)

# ===================================
# Declare some intermediate variables
# ===================================
pi = np.pi
C_D_fuse = CDA0 / S
C_D_wpar = k * C_f * S_wetratio
C_D_ind = C_L**2 / (pi * A * e)
W_w_strc = W_W_coeff1 * (N_ult * A**1.5 * (W_0 * W * S) ** 0.5) / tau
W_w_surf = W_W_coeff2 * S

# =======================
# Declare the Constraints
# =======================
f.ConstraintList(
    [
        C_D >= C_D_fuse + C_D_wpar + C_D_ind,
        W_w >= W_w_surf + W_w_strc ,
        D >= 0.5 * rho * S * C_D * V**2,
        Re == (rho / mu) * V * (S / A) ** 0.5,
        C_f == 0.074 / Re**0.2,
        W == 0.5 * rho * S * C_L * V**2,
        W == 0.5 * rho * S * C_Lmax * V_min**2,
        W >= W_0 + W_w,
    ])

# =======================
# Print Model
# =======================
# f.pprint()

# from pyomo.environ import SolverFactory
# opt = SolverFactory('ipopt')
# opt.solve(f)
# # print('The drag of the aircraft is %f N'%(f.D.value))

from edi.solvers.solver import cvxopt_solve
res = cvxopt_solve(f)
print(res)
print(res['x'])


lbls = [
    'straight_1',
    'right_turn_1',
    'straight_2',
    'left_turn',
    'straight_3',
    'right_turn_2',
    'straight_4'
]
var_list = f.get_variables()
ctr = 0
pstr = ''
for var in var_list:
    if isinstance(var,pyomo.core.base.var.IndexedVar):
        for ix in var.index_set():
            pstr += '%s[%s]:  %.4f  %s\n'%(var.name,lbls[ix],res['x'][ctr],var._units)
            ctr += 1
    else:
        pstr += '%s:  %.4f  %s\n'%(var.name,res['x'][ctr],var._units)
        ctr += 1
mxln = 0
for line in pstr.split('\n'):
    lhc = line.split(':')[0]
    if len(lhc)>mxln:
        mxln = len(lhc)
pstr2 = ''
for line in pstr.split('\n'):
    lhc = line.split(':')[0]
    pstr2 += line.replace(lhc,lhc.ljust(mxln+2)) + '\n'
print(pstr2)

# import pyomo
# var_list = f.get_variables()
# for var in var_list:
#     if isinstance(var,pyomo.core.base.var.IndexedVar):
#         for ix in var.index_set():
#             lbls = ["out","ret","sprint"]
#             print('%s[%s]:  %.4f  %s'%(var.name,lbls[ix],var[ix].value,var._units))
#     else:
#         print('%s:  %.4f  %s'%(var.name,var.value,var._units))

{'status': 'optimal', 'primal objective': 254.8707175492974, 'dual objective': 254.87370600273985, 'x': <10x1 matrix, tc='d'>, 'y': <4x1 matrix, tc='d'>, 'z': <13x1 matrix, tc='d'>, 's': <12x1 matrix, tc='d'>, 'N_cons_total': 8, 'N_cons_noBounds': 8, 'N_cons_bounds': 0, 'inequality_unscramble': [0, 1, 2, 3, 8], 'equality_unscramble': [4, 5, 6, 7], 'problem_structure': 'geometric_program'}
[ 1.27e+01]
[ 2.31e-02]
[ 3.86e-03]
[ 6.51e-01]
[ 2.55e+02]
[ 2.60e+06]
[ 1.30e+02]
[ 3.86e+01]
[ 7.19e+03]
[ 2.25e+03]

A    :  12.7003  dimensionless
C_D  :  0.0231  dimensionless
C_f  :  0.0039  dimensionless
C_L  :  0.6513  dimensionless
D    :  254.8707  N
Re   :  2597080.1556  dimensionless
S    :  129.9324  ft**2
V    :  38.5508  m/s
W    :  7186.1774  N
W_w  :  2246.2219  N
     

